In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv

workspace_root = Path.cwd().parent
env_paths = (
    Path.cwd() / ".env",
    workspace_root / ".env",
    workspace_root / "Langchain_Basics" / ".env",
)

for env_path in env_paths:
    if env_path.exists():
        load_dotenv(env_path, override=True)
        print(f"Loaded environment from: {env_path}")
        break
else:
    print("No .env file found. Create Agents/.env or workspace-root/.env.")

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "LangChainTrainings-Agents")

if os.getenv("LANGSMITH_API_KEY"):
    print(f"LangSmith tracing enabled for project: {os.environ['LANGSMITH_PROJECT']}")
else:
    print("Add LANGSMITH_API_KEY to .env to enable LangSmith tracing.")

Loaded environment from: c:\Users\Girish Kulkarni\Downloads\LangChainTrainings\Langchain_Basics\.env
LangSmith tracing enabled for project: Firstproject


In [2]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen3:8b",
    temperature=0.5,
    num_predict=2500,
    reasoning=False,
)

In [3]:
# Wikipedia tool with retry handling

import json
import time

from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun

wikipedia = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(
        top_k_results=2,
        doc_content_chars_max=4000,
    )
)


def wikipedia_invoke_with_retry(query, max_attempts=3):
    for attempt in range(max_attempts):
        try:
            return wikipedia.invoke(query)
        except (json.JSONDecodeError, ConnectionError, TimeoutError) as error:
            if attempt == max_attempts - 1:
                return f"Wikipedia request failed after {max_attempts} attempts: {error}"
            time.sleep(2 ** attempt)


tool_response = wikipedia_invoke_with_retry("What is the capital of India?")
tool_response

C:\Users\Girish Kulkarni\AppData\Local\Temp\ipykernel_4740\3382639280.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import WikipediaAPIWrapper


'Wikipedia request failed after 3 attempts: Expecting value: line 1 column 1 (char 0)'

In [4]:
# Simple DuckDuckGo search

from langchain_community.tools import DuckDuckGoSearchRun

duckduckgo_search = DuckDuckGoSearchRun()

search_result = duckduckgo_search.invoke("Where is the Eiffel Tower located?")
print(search_result)

August 6, 2026 - Tokyo Tower in Japan, built as ... by the Eiffel Tower. Well known is the Petřín Lookout Tower in Prague too. There are various scale models of the tower in the United States, including a half-scale version at the Paris Las Vegas, Nevada, one in Paris, Texas built in 1993, and two 1:3 scale models at Kings Island, located in Cincinnati, ... May 2, 2026 - The Eiffel Tower (French: La Tour Eiffel, [tuʁ ɛfɛl], IPA pronunciation: "EYE-full" English; "Eiffel" French) is a wrought-iron landmark in Paris. It was built between 1887 and 1889 for the Exposition Universelle (World Fair). It was supposed to be a temporary installation for the 1889 World ... August 8, 2026 - Eiffel Tower, wrought-iron structure in Paris that is one of the most famous landmarks in the world. It is also a technological masterpiece in building-construction history. It was designed and built (1887–89) by Gustave Eiffel and named in ... June 20, 2026 - The Eiffel Tower is located in the heart of Paris, 

In [5]:
# Creatin custome tools

from langchain.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

@tool
def substarct(a: int, b: int) -> int:
    """Add two numbers."""
    return a - b

@tool
def multiply(a: int, b: int) -> int:
    """Add two numbers."""
    return a * b

print(add.invoke({"a":10, "b": 20}))  # Example usage of the custom tool



30


In [6]:
tools = [wikipedia, add, substarct, multiply]

# list_of_tools = {tool.name: tool for tool in tools}

llm_with_tools = llm.bind_tools(tools)

response = llm_with_tools.invoke(
    "What is the capital of India"
)

response

AIMessage(content='The capital of India is New Delhi.', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-09-08T09:48:00.4691222Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1573774800, 'load_duration': 7920600, 'prompt_eval_count': 310, 'prompt_eval_duration': 327269000, 'eval_count': 9, 'eval_duration': 1224070000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--01a0806a-c82c-7cc1-bd9a-6ae322c32470-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 310, 'output_tokens': 9, 'total_tokens': 319})